# Day21A–Day23A Cross-Audit Synthesis

## Scope

This notebook synthesizes the closed Day21A, Day22A, and Day23A MJ1 audit outputs.

The purpose is not to recompute trajectories, residuals, or verdicts, but to consolidate already-closed audit results into a single comparison table.

## Included audit branches

### Day21A

High-amplitude AC-off protocol family:

- 0.3C DC reference
- 0.3C + 0.7C 0.1τ
- 0.3C + 0.7C 1τ
- 0.3C + 0.7C 10τ

Day21A uses the original Segment A/B/D framework because AC is switched off after Vmax.

### Day22A

Lower-amplitude AC-off protocol family:

- 0.3C DC reference
- 0.3C + 0.4C 0.1τ
- 0.3C + 0.4C 1τ
- 0.3C + 0.4C 10τ

Day22A also uses the original Segment A/B/D framework because AC is switched off after Vmax.

### Day23A

Sub-DC-amplitude protocol-mode contrast:

- 0.4C DC reference
- 0.4C + 0.1C 1τ
- 0.4C + 0.3C 1τ

0.4C + 0.2C 1τ is excluded because the raw file lacks the required pre-Vmax CC segment.

Day23A does not use the original Segment A/B/D framework because the active DC–AC files retain substantial AC components after Vmax.

Day23A therefore uses the generalized G0/G1/G2 boundary-ordering framework.

## Core comparison dimensions

The synthesis table compares:

- protocol family;
- DC_C, AC_C, and κ;
- AC-off after Vmax status;
- boundary ordering;
- Q80/Q90 common region;
- Q80/Q90 common raw Δt;
- Segment-A or G0 residual status;
- fitted residual diagnostic status;
- final-Q caveats;
- closed verdict.

## Interpretation rule

Raw Δt(Q) is a real first-passage time difference, but it is not mechanism-pure.

Day21A and Day22A support a boundary/control-state mediated interpretation under AC-off protocols.

Day23A is a different protocol-mode family and shows that boundary-leading does not necessarily imply full-protocol gain preservation.

## Non-goals

This notebook does not:

- recompute Q integration;
- recompute event detection;
- recompute Δt(Q);
- redefine any threshold;
- claim confirmed non-geometric Segment-A or G0 electrochemical acceleration.

## Key conclusion

The Day21A–Day23A audits jointly show that earlier Vmax triggering under DC–AC excitation can be a real boundary event. However, boundary-leading does not automatically imply full-protocol state-equivalent gain preservation, and it does not constitute confirmed non-geometric Segment-A/G0 electrochemical acceleration.

In [1]:
# Notebook 28 Cell 1 — setup and source file inventory
#
# Purpose:
# - Define input files from Day21A / Day22A / Day23A
# - Check that all required closed-audit outputs exist
# - Write a source inventory for traceability
#
# Explicitly NOT done here:
# - No recomputation
# - No trajectory loading
# - No Q integration
# - No verdict modification

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import numpy as np
import pandas as pd

REPO = Path("/Users/louislu/pybamm-dcac-superimposed")
DATA_DIR = REPO / "data"
NOTEBOOK_NAME = "28_day21A_day22A_day23A_cross_audit_synthesis.ipynb"

OUT_CROSS_SOURCE_INVENTORY = DATA_DIR / "day21A_day22A_day23A_cross_audit_source_inventory.csv"
OUT_CROSS_SYNTHESIS_TABLE = DATA_DIR / "day21A_day22A_day23A_cross_audit_synthesis_table.csv"
OUT_CROSS_SYNTHESIS_MD = DATA_DIR / "day21A_day22A_day23A_cross_audit_synthesis_note.md"

def git_head_or_unknown(repo_path: Path):
    try:
        out = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_path,
            stderr=subprocess.DEVNULL,
        )
        return out.decode("utf-8").strip()
    except Exception:
        return "unknown_git_head"

GIT_HEAD = git_head_or_unknown(REPO)

# =============================================================================
# Input files
# =============================================================================

INPUTS = {
    # Day21A
    "day21A_closure": DATA_DIR / "day21A_step8_closure_summary.csv",
    "day21A_verdict": DATA_DIR / "day21A_step6_MJ1_mechanism_verdict.csv",
    "day21A_segment_assignment": DATA_DIR / "day21A_step4_MJ1_segment_assignment.csv",
    "day21A_dtq_summary": DATA_DIR / "day21A_step5_MJ1_dtQ_segment_summary.csv",
    "day21A_diagnostic": DATA_DIR / "day21A_step5E_MJ1_residual_diagnostic_synthesis.csv",

    # Day22A
    "day22A_closure": DATA_DIR / "day22A_step7_closure_summary.csv",
    "day22A_verdict": DATA_DIR / "day22A_step6_MJ1_0p3C_0p4C_mechanism_verdict.csv",
    "day22A_segment_assignment": DATA_DIR / "day22A_step4_MJ1_0p3C_0p4C_segment_assignment.csv",
    "day22A_dtq_summary": DATA_DIR / "day22A_step5_MJ1_0p3C_0p4C_dtQ_segment_summary.csv",
    "day22A_resolution": DATA_DIR / "day22A_step3A_MJ1_0p3C_0p4C_resolution_floor_summary.csv",

    # Day23A
    "day23A_closure": DATA_DIR / "day23A_step7_closure_summary.csv",
    "day23A_verdict": DATA_DIR / "day23A_step6_MJ1_0p4C_subDC_generalized_verdict.csv",
    "day23A_g_assignment": DATA_DIR / "day23A_step4_MJ1_0p4C_subDC_G0G1G2_assignment.csv",
    "day23A_dtq_summary": DATA_DIR / "day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_summary.csv",
    "day23A_resolution": DATA_DIR / "day23A_step3A_MJ1_0p4C_subDC_resolution_floor_summary.csv",
    "day23A_framework": DATA_DIR / "day23A_step0D_framework_decision.csv",
    "day23A_exclusion": DATA_DIR / "day23A_step0A5_file_exclusion_audit.csv",
}

rows = []

for key, path in INPUTS.items():
    rows.append({
        "source_key": key,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan,
    })

df_source_inventory = pd.DataFrame(rows)
df_source_inventory.to_csv(OUT_CROSS_SOURCE_INVENTORY, index=False)

print(f"[OK] Notebook = {NOTEBOOK_NAME}")
print(f"[OK] Repo = {REPO}")
print(f"[OK] Git HEAD = {GIT_HEAD}")
print(f"[OK] Wrote source inventory: {OUT_CROSS_SOURCE_INVENTORY}")
print(df_source_inventory.to_string(index=False))

missing = df_source_inventory[~df_source_inventory["exists"]]

if len(missing) > 0:
    print("\n[ERROR] Missing required source files:")
    print(missing.to_string(index=False))
    raise FileNotFoundError("Cross-audit synthesis cannot proceed because required files are missing.")

print("[OK] All required Day21A / Day22A / Day23A source files exist.")
print("[OK] No recomputation performed in Cell 1.")

[OK] Notebook = 28_day21A_day22A_day23A_cross_audit_synthesis.ipynb
[OK] Repo = /Users/louislu/pybamm-dcac-superimposed
[OK] Git HEAD = d255cd1dbd4598d6e2a499864a1aca37c3dd7cb9
[OK] Wrote source inventory: /Users/louislu/pybamm-dcac-superimposed/data/day21A_day22A_day23A_cross_audit_source_inventory.csv
               source_key                                                                                                   path  exists  size_bytes
           day21A_closure                          /Users/louislu/pybamm-dcac-superimposed/data/day21A_step8_closure_summary.csv    True        3080
           day21A_verdict                    /Users/louislu/pybamm-dcac-superimposed/data/day21A_step6_MJ1_mechanism_verdict.csv    True        7422
day21A_segment_assignment                   /Users/louislu/pybamm-dcac-superimposed/data/day21A_step4_MJ1_segment_assignment.csv    True        2632
       day21A_dtq_summary                  /Users/louislu/pybamm-dcac-superimposed/data/day21A_step

In [2]:
# Notebook 28 Cell 2 — normalize Day21A / Day22A / Day23A into one synthesis table
#
# Purpose:
# - Load closed audit outputs from Day21A / Day22A / Day23A
# - Normalize them into one cross-audit synthesis table
# - Preserve framework differences:
#   Day21A/Day22A = AC-off A/B/D family
#   Day23A = continued-AC G0/G1/G2 family
#
# Explicitly NOT done here:
# - No Q integration
# - No event detection
# - No Δt recomputation
# - No verdict modification

# =============================================================================
# 2.1 Helpers
# =============================================================================

def read_csv_key(key):
    path = INPUTS[key]
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {key} -> {path}")
    return pd.read_csv(path)


def row_by_pair(df, pair, label):
    if "protocol_pair" not in df.columns:
        return pd.Series(dtype=object)

    rows = df[df["protocol_pair"] == pair]
    if len(rows) == 0:
        return pd.Series(dtype=object)

    if len(rows) > 1:
        raise ValueError(f"{label}: expected one row for {pair}, found {len(rows)}")

    return rows.iloc[0]


def pick(row, candidates, default=np.nan):
    for c in candidates:
        if c in row.index:
            return row[c]
    return default


def pick_str(row, candidates, default=""):
    val = pick(row, candidates, default=default)
    if pd.isna(val):
        return default
    return str(val)


def pick_float(row, candidates, default=np.nan):
    val = pick(row, candidates, default=default)
    try:
        return float(val)
    except Exception:
        return default


def parse_protocol_pair(pair):
    """
    Parse strings like:
    - 0.3C DC vs 0.3C+0.7C 10tau
    - 0.4C DC vs 0.4C+0.3C 1tau
    """
    import re

    s = str(pair)

    dc_ref = np.nan
    dc_c = np.nan
    ac_c = np.nan
    m_tau = np.nan

    m_ref = re.search(r"(\d+(?:\.\d+)?)C\s*DC", s)
    if m_ref:
        dc_ref = float(m_ref.group(1))

    m_dcac = re.search(
        r"vs\s+(\d+(?:\.\d+)?)C\+(\d+(?:\.\d+)?)C\s+(\d+(?:\.\d+)?)tau",
        s,
    )
    if m_dcac:
        dc_c = float(m_dcac.group(1))
        ac_c = float(m_dcac.group(2))
        m_tau = float(m_dcac.group(3))

    kappa = ac_c / dc_c if np.isfinite(ac_c) and np.isfinite(dc_c) and dc_c != 0 else np.nan

    return {
        "DC_reference_C": dc_ref,
        "DC_C": dc_c,
        "AC_C": ac_c,
        "kappa": kappa,
        "m_tau": m_tau,
    }


def boundary_order_from_shift(q_shift_ah, tol_ah=0.001):
    """
    q_shift = Q_Vmax_DC - Q_Vmax_DCAC.
    Positive means DCAC reaches Vmax at lower Q.
    """
    if not np.isfinite(q_shift_ah):
        return "boundary_order_unresolved"

    if q_shift_ah > tol_ah:
        return "DCAC_first"

    if q_shift_ah < -tol_ah:
        return "DC_first"

    return "boundary_degenerate"


def normalize_abd_day(day_label, amplitude_class, closure_key, verdict_key, seg_key, dtq_key, diagnostic_key=None):
    """
    Normalize Day21A / Day22A style AC-off A/B/D audit.
    """
    closure = read_csv_key(closure_key)
    verdict = read_csv_key(verdict_key)
    seg = read_csv_key(seg_key)
    dtq = read_csv_key(dtq_key)

    diagnostic = read_csv_key(diagnostic_key) if diagnostic_key is not None else pd.DataFrame()

    rows = []

    for _, c in closure.iterrows():
        pair = c["protocol_pair"]
        p = parse_protocol_pair(pair)

        v = row_by_pair(verdict, pair, f"{day_label}:verdict")
        s = row_by_pair(seg, pair, f"{day_label}:segment")
        d = row_by_pair(dtq, pair, f"{day_label}:dtq")
        dg = row_by_pair(diagnostic, pair, f"{day_label}:diagnostic") if diagnostic_key is not None else pd.Series(dtype=object)

        q_shift = pick_float(s, ["Q_Vmax_shift_Ah", "Q_Vmax_shift_Ah_DC_minus_DCAC"])
        boundary_order = boundary_order_from_shift(q_shift)

        prescribed_p95 = pick_float(
            d,
            ["segment_A_dt_resid_p95_abs_s", "G0_dt_resid_p95_abs_s"],
            default=np.nan,
        )
        if not np.isfinite(prescribed_p95):
            prescribed_p95 = pick_float(
                c,
                ["segment_A_dt_resid_p95_abs_s", "G0_dt_resid_p95_abs_s"],
                default=np.nan,
            )

        fitted_p95 = pick_float(
            c,
            ["segment_A_dt_resid_fit_p95_abs_s", "G0_dt_resid_fit_p95_abs_s"],
            default=np.nan,
        )
        if not np.isfinite(fitted_p95):
            fitted_p95 = pick_float(
                dg,
                ["dt_resid_fitted_p95_abs_s", "segment_A_dt_resid_fit_p95_abs_s"],
                default=np.nan,
            )

        rows.append({
            "audit_day": day_label,
            "protocol_family": "AC_off_after_Vmax",
            "framework": "A_B_D_AC_off_segmentation",
            "amplitude_class": amplitude_class,

            "protocol_pair": pair,
            "DC_reference_C": p["DC_reference_C"],
            "DC_C": p["DC_C"],
            "AC_C": p["AC_C"],
            "kappa": p["kappa"],
            "m_tau": p["m_tau"],

            "AC_off_after_Vmax_status": "supported_by_audit",
            "post_Vmax_AC_status": "AC_off_after_Vmax",
            "same_protocol_family_as_Day21A_Day22A": True,

            "boundary_ordering_by_Q": boundary_order,
            "Q_Vmax_shift_Ah_DC_minus_DCAC": q_shift,

            "Q80_common_region": pick_str(c, ["Q80_common_segment"], default=""),
            "Q90_common_region": pick_str(c, ["Q90_common_segment"], default=""),

            "dt_Q80_common_raw_s": pick_float(c, ["dt_Q80_common_raw_s"]),
            "dt_Q90_common_raw_s": pick_float(c, ["dt_Q90_common_raw_s"]),

            "residual_region_name": "Segment_A",
            "prescribed_residual_status": pick_str(
                c,
                ["segment_A_above_floor_status"],
                default="",
            ),
            "prescribed_residual_p95_abs_s": prescribed_p95,
            "fitted_residual_p95_abs_s": fitted_p95,

            "audit_resolution_p95_s": np.nan,
            "Q80_raw_resolution_status": "",
            "Q90_raw_resolution_status": "",

            "final_Q_status": pick_str(s, ["Q_final_diff_status"], default=""),
            "evidence_status": pick_str(c, ["evidence_status"], default=pick_str(v, ["evidence_status"], default="")),
            "mechanism_verdict": pick_str(c, ["mechanism_verdict"], default=pick_str(v, ["mechanism_verdict"], default="")),
            "interpretation_class": pick_str(c, ["interpretation_class"], default=pick_str(v, ["interpretation_class"], default="")),
            "caveat": pick_str(c, ["caveat"], default=pick_str(v, ["caveat"], default="")),

            "synthesis_interpretation": "",
        })

    return pd.DataFrame(rows)


def normalize_day23():
    """
    Normalize Day23A generalized G0/G1/G2 audit.
    """
    closure = read_csv_key("day23A_closure")
    verdict = read_csv_key("day23A_verdict")
    g = read_csv_key("day23A_g_assignment")
    resolution = read_csv_key("day23A_resolution")

    resolution_p95 = float(resolution["day23A_self_consistency_resolution_p95_s"].iloc[0])

    rows = []

    for _, c in closure.iterrows():
        pair = c["protocol_pair"]
        p = parse_protocol_pair(pair)

        v = row_by_pair(verdict, pair, "Day23A:verdict")
        gg = row_by_pair(g, pair, "Day23A:g_assignment")

        rows.append({
            "audit_day": "Day23A",
            "protocol_family": "continued_AC_after_Vmax",
            "framework": "G0_G1_G2_generalized_boundary_ordering",
            "amplitude_class": "sub_DC_amplitude",

            "protocol_pair": pair,
            "DC_reference_C": p["DC_reference_C"],
            "DC_C": pick_float(v, ["DC_C"], default=p["DC_C"]),
            "AC_C": pick_float(v, ["AC_C"], default=p["AC_C"]),
            "kappa": pick_float(c, ["kappa"], default=p["kappa"]),
            "m_tau": pick_float(v, ["m_tau"], default=p["m_tau"]),

            "AC_off_after_Vmax_status": "not_supported",
            "post_Vmax_AC_status": pick_str(c, ["protocol_mode_status_DCAC"], default="possible_full_DCAC_after_Vmax"),
            "same_protocol_family_as_Day21A_Day22A": False,

            "boundary_ordering_by_Q": pick_str(c, ["boundary_ordering_by_Q"], default=pick_str(gg, ["boundary_ordering_by_Q"], default="")),
            "Q_Vmax_shift_Ah_DC_minus_DCAC": pick_float(
                v,
                ["Q_Vmax_shift_Ah_DC_minus_DCAC"],
                default=pick_float(gg, ["Q_Vmax_shift_Ah_DC_minus_DCAC"], default=np.nan),
            ),

            "Q80_common_region": pick_str(c, ["Q80_common_region"], default=""),
            "Q90_common_region": pick_str(c, ["Q90_common_region"], default=""),

            "dt_Q80_common_raw_s": pick_float(c, ["dt_Q80_common_raw_s"]),
            "dt_Q90_common_raw_s": pick_float(c, ["dt_Q90_common_raw_s"]),

            "residual_region_name": "G0",
            "prescribed_residual_status": pick_str(c, ["G0_residual_status"], default=""),
            "prescribed_residual_p95_abs_s": pick_float(c, ["G0_dt_resid_p95_abs_s"]),
            "fitted_residual_p95_abs_s": pick_float(c, ["G0_dt_resid_fit_p95_abs_s"]),

            "audit_resolution_p95_s": resolution_p95,
            "Q80_raw_resolution_status": pick_str(c, ["Q80_raw_resolution_status"], default=""),
            "Q90_raw_resolution_status": pick_str(c, ["Q90_raw_resolution_status"], default=""),

            "final_Q_status": pick_str(c, ["Q_final_diff_status"], default=""),
            "evidence_status": pick_str(c, ["evidence_status"], default=pick_str(v, ["evidence_status"], default="")),
            "mechanism_verdict": pick_str(c, ["mechanism_verdict"], default=pick_str(v, ["mechanism_verdict"], default="")),
            "interpretation_class": pick_str(c, ["interpretation_class"], default=pick_str(v, ["interpretation_class"], default="")),
            "caveat": pick_str(c, ["caveat"], default=pick_str(v, ["caveat"], default="")),

            "synthesis_interpretation": "",
        })

    return pd.DataFrame(rows)


# =============================================================================
# 2.2 Normalize all audits
# =============================================================================

df_day21_norm = normalize_abd_day(
    day_label="Day21A",
    amplitude_class="high_amplitude_AC_gt_DC",
    closure_key="day21A_closure",
    verdict_key="day21A_verdict",
    seg_key="day21A_segment_assignment",
    dtq_key="day21A_dtq_summary",
    diagnostic_key="day21A_diagnostic",
)

df_day22_norm = normalize_abd_day(
    day_label="Day22A",
    amplitude_class="lower_amplitude_AC_gt_DC",
    closure_key="day22A_closure",
    verdict_key="day22A_verdict",
    seg_key="day22A_segment_assignment",
    dtq_key="day22A_dtq_summary",
    diagnostic_key=None,
)

df_day23_norm = normalize_day23()

df_cross = pd.concat(
    [df_day21_norm, df_day22_norm, df_day23_norm],
    ignore_index=True,
)


# =============================================================================
# 2.3 Add synthesis-level interpretation labels
# =============================================================================

def synthesize_interpretation(row):
    day = row["audit_day"]
    family = row["protocol_family"]

    q80 = row["dt_Q80_common_raw_s"]
    q90 = row["dt_Q90_common_raw_s"]
    kappa = row["kappa"]

    if day == "Day21A":
        if row["Q80_common_region"].startswith("B") and row["Q90_common_region"].startswith("B"):
            return "high_amplitude_AC_off_boundary_control_state_dominant"
        if row["Q80_common_region"].startswith("A") and row["Q90_common_region"].startswith("B"):
            return "mixed_A_to_B_transition_under_high_amplitude"
        return "high_amplitude_AC_off_mixed"

    if day == "Day22A":
        if row["Q80_common_region"].startswith("A") and row["Q90_common_region"].startswith("B"):
            return "lower_amplitude_weakens_boundary_pathway_Q80_returns_to_A"
        return "lower_amplitude_AC_off_mixed"

    if day == "Day23A":
        if row["interpretation_class"] == "boundary_leading_not_gain_preserving":
            return "continued_AC_boundary_leading_but_not_gain_preserving"
        if row["interpretation_class"] == "subDC_small_perturbation_near_resolution":
            return "continued_AC_subDC_gain_unresolved_near_resolution"
        return "continued_AC_generalized_boundary_result"

    return ""


df_cross["synthesis_interpretation"] = df_cross.apply(synthesize_interpretation, axis=1)

# Sorting: Day21A, Day22A, Day23A; then m_tau / kappa
day_order = {"Day21A": 1, "Day22A": 2, "Day23A": 3}
df_cross["_day_order"] = df_cross["audit_day"].map(day_order)
df_cross = df_cross.sort_values(
    by=["_day_order", "m_tau", "kappa"],
    ascending=[True, True, True],
).drop(columns=["_day_order"])

df_cross.to_csv(OUT_CROSS_SYNTHESIS_TABLE, index=False)

print(f"[OK] Wrote cross-audit synthesis table: {OUT_CROSS_SYNTHESIS_TABLE}")
print(f"[OK] synthesis rows = {len(df_cross)}")

display_cols = [
    "audit_day",
    "protocol_pair",
    "protocol_family",
    "framework",
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "AC_off_after_Vmax_status",
    "boundary_ordering_by_Q",
    "Q_Vmax_shift_Ah_DC_minus_DCAC",
    "Q80_common_region",
    "Q90_common_region",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "prescribed_residual_status",
    "fitted_residual_p95_abs_s",
    "evidence_status",
    "mechanism_verdict",
    "synthesis_interpretation",
]

print(df_cross[display_cols].to_string(index=False))

# Hard guards
expected_rows = 3 + 3 + 2
if len(df_cross) != expected_rows:
    raise ValueError(f"Expected {expected_rows} synthesis rows, got {len(df_cross)}")

if (df_cross[df_cross["audit_day"].isin(["Day21A", "Day22A"])]["same_protocol_family_as_Day21A_Day22A"] != True).any():
    raise ValueError("Day21A/Day22A protocol-family flag inconsistent.")

if (df_cross[df_cross["audit_day"] == "Day23A"]["same_protocol_family_as_Day21A_Day22A"] != False).any():
    raise ValueError("Day23A protocol-family flag inconsistent.")

print("[OK] Cell 2 cross-audit normalization completed.")
print("[OK] No original audit verdict was modified.")

[OK] Wrote cross-audit synthesis table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_day22A_day23A_cross_audit_synthesis_table.csv
[OK] synthesis rows = 11
audit_day                                               protocol_pair         protocol_family                              framework  DC_C  AC_C    kappa  m_tau AC_off_after_Vmax_status    boundary_ordering_by_Q  Q_Vmax_shift_Ah_DC_minus_DCAC                      Q80_common_region                      Q90_common_region  dt_Q80_common_raw_s  dt_Q90_common_raw_s                      prescribed_residual_status  fitted_residual_p95_abs_s                         evidence_status                                                                   mechanism_verdict                                  synthesis_interpretation
   Day21A                                 0.3C DC vs 0.3C+0.7C 0.1tau       AC_off_after_Vmax              A_B_D_AC_off_segmentation   0.3   0.7 2.333333    0.1       supported_by_audit                DCAC_first      

ValueError: Expected 8 synthesis rows, got 11

In [3]:
# Notebook 28 Cell 2B — filter MJ1 experimental main synthesis table
#
# Purpose:
# - Remove PyBaMM Day20B context rows from the main Day21A–Day23A synthesis table
# - Preserve excluded PyBaMM context rows in a separate file
# - Overwrite the main synthesis table with MJ1 experimental rows only
#
# Reason:
# - Day21A closure contains 3 MJ1 rows + 3 PyBaMM context rows
# - Cross-audit main table should contain only MJ1 experimental Day21A/22A/23A rows

OUT_CROSS_PYBAMM_CONTEXT = DATA_DIR / "day21A_day22A_day23A_cross_audit_excluded_pybamm_context.csv"

if "df_cross" not in globals():
    if not OUT_CROSS_SYNTHESIS_TABLE.exists():
        raise FileNotFoundError(f"Missing synthesis table: {OUT_CROSS_SYNTHESIS_TABLE}")
    df_cross = pd.read_csv(OUT_CROSS_SYNTHESIS_TABLE)

# MJ1 experimental rows have parsed finite DC_C / AC_C / kappa.
# PyBaMM context rows from Day21A have NaN DC_C / AC_C / kappa in this normalization.
mask_mj1_experimental = (
    df_cross["DC_C"].notna()
    & df_cross["AC_C"].notna()
    & df_cross["kappa"].notna()
)

df_cross_main = df_cross.loc[mask_mj1_experimental].copy()
df_cross_context_excluded = df_cross.loc[~mask_mj1_experimental].copy()

# Save context rows separately for traceability.
df_cross_context_excluded.to_csv(OUT_CROSS_PYBAMM_CONTEXT, index=False)

# Overwrite main synthesis table with MJ1-only rows.
df_cross_main.to_csv(OUT_CROSS_SYNTHESIS_TABLE, index=False)

print(f"[OK] Wrote MJ1-only main synthesis table: {OUT_CROSS_SYNTHESIS_TABLE}")
print(f"[OK] Wrote excluded PyBaMM context table: {OUT_CROSS_PYBAMM_CONTEXT}")
print(f"[OK] main rows = {len(df_cross_main)}")
print(f"[OK] excluded context rows = {len(df_cross_context_excluded)}")

print("\n[Main MJ1 experimental synthesis]")
display_cols = [
    "audit_day",
    "protocol_pair",
    "protocol_family",
    "framework",
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "AC_off_after_Vmax_status",
    "boundary_ordering_by_Q",
    "Q_Vmax_shift_Ah_DC_minus_DCAC",
    "Q80_common_region",
    "Q90_common_region",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "prescribed_residual_status",
    "fitted_residual_p95_abs_s",
    "evidence_status",
    "mechanism_verdict",
    "synthesis_interpretation",
]

print(df_cross_main[display_cols].to_string(index=False))

print("\n[Excluded PyBaMM context rows]")
if len(df_cross_context_excluded) > 0:
    print(df_cross_context_excluded[[
        "audit_day",
        "protocol_pair",
        "protocol_family",
        "DC_C",
        "AC_C",
        "kappa",
        "mechanism_verdict",
    ]].to_string(index=False))
else:
    print("None")

# Hard guards
expected_rows = 8
if len(df_cross_main) != expected_rows:
    raise ValueError(f"Expected {expected_rows} MJ1 experimental synthesis rows, got {len(df_cross_main)}")

expected_counts = {
    "Day21A": 3,
    "Day22A": 3,
    "Day23A": 2,
}

actual_counts = df_cross_main["audit_day"].value_counts().to_dict()

for day, expected in expected_counts.items():
    actual = actual_counts.get(day, 0)
    if actual != expected:
        raise ValueError(f"{day}: expected {expected} rows, got {actual}")

if len(df_cross_context_excluded) != 3:
    raise ValueError(
        f"Expected 3 excluded PyBaMM context rows from Day21A, got {len(df_cross_context_excluded)}"
    )

print("[OK] Cell 2B completed.")
print("[OK] Main synthesis table now contains MJ1 experimental rows only.")
print("[OK] PyBaMM context preserved separately, not discarded.")

[OK] Wrote MJ1-only main synthesis table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_day22A_day23A_cross_audit_synthesis_table.csv
[OK] Wrote excluded PyBaMM context table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_day22A_day23A_cross_audit_excluded_pybamm_context.csv
[OK] main rows = 8
[OK] excluded context rows = 3

[Main MJ1 experimental synthesis]
audit_day               protocol_pair         protocol_family                              framework  DC_C  AC_C    kappa  m_tau AC_off_after_Vmax_status boundary_ordering_by_Q  Q_Vmax_shift_Ah_DC_minus_DCAC                      Q80_common_region                      Q90_common_region  dt_Q80_common_raw_s  dt_Q90_common_raw_s                      prescribed_residual_status  fitted_residual_p95_abs_s                        evidence_status                                                   mechanism_verdict                                  synthesis_interpretation
   Day21A 0.3C DC vs 0.3C+0.7C 0.1tau       AC_off_after_Vm

In [4]:
# Notebook 28 Cell 3 — write cross-audit synthesis note
#
# Purpose:
# - Generate a human-readable synthesis note from the MJ1-only cross-audit table
# - Preserve PyBaMM context separately
# - State allowed and prohibited conclusions
#
# Explicitly NOT done here:
# - No recomputation
# - No verdict modification
# - No new mechanism claim

if not OUT_CROSS_SYNTHESIS_TABLE.exists():
    raise FileNotFoundError(f"Missing main synthesis table: {OUT_CROSS_SYNTHESIS_TABLE}")

OUT_CROSS_PYBAMM_CONTEXT = DATA_DIR / "day21A_day22A_day23A_cross_audit_excluded_pybamm_context.csv"

df_cross_main = pd.read_csv(OUT_CROSS_SYNTHESIS_TABLE)

if OUT_CROSS_PYBAMM_CONTEXT.exists():
    df_pybamm_context = pd.read_csv(OUT_CROSS_PYBAMM_CONTEXT)
else:
    df_pybamm_context = pd.DataFrame()

expected_counts = {
    "Day21A": 3,
    "Day22A": 3,
    "Day23A": 2,
}

actual_counts = df_cross_main["audit_day"].value_counts().to_dict()

for day, expected in expected_counts.items():
    actual = actual_counts.get(day, 0)
    if actual != expected:
        raise ValueError(f"{day}: expected {expected} rows, got {actual}")

summary_cols = [
    "audit_day",
    "protocol_pair",
    "protocol_family",
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "AC_off_after_Vmax_status",
    "boundary_ordering_by_Q",
    "Q_Vmax_shift_Ah_DC_minus_DCAC",
    "Q80_common_region",
    "Q90_common_region",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "mechanism_verdict",
    "synthesis_interpretation",
]

summary_table = df_cross_main[summary_cols].to_string(index=False)

day21_table = df_cross_main[df_cross_main["audit_day"] == "Day21A"][summary_cols].to_string(index=False)
day22_table = df_cross_main[df_cross_main["audit_day"] == "Day22A"][summary_cols].to_string(index=False)
day23_table = df_cross_main[df_cross_main["audit_day"] == "Day23A"][summary_cols].to_string(index=False)

if len(df_pybamm_context) > 0:
    pybamm_table = df_pybamm_context[[
        "audit_day",
        "protocol_pair",
        "protocol_family",
        "mechanism_verdict",
    ]].to_string(index=False)
else:
    pybamm_table = "None"

now_utc = datetime.now(timezone.utc).isoformat()

lines = []

lines.append("# Day21A–Day23A Cross-Audit Synthesis")
lines.append("")
lines.append(f"Generated: `{now_utc}`")
lines.append(f"Git HEAD: `{GIT_HEAD}`")
lines.append(f"Notebook: `{NOTEBOOK_NAME}`")
lines.append("")
lines.append("## 1. Scope")
lines.append("")
lines.append("This note synthesizes the closed MJ1 experimental audits Day21A, Day22A, and Day23A.")
lines.append("")
lines.append("The purpose is to compare already-closed audit results. No trajectory, Q integration, event timing, Δt(Q), or verdict is recomputed here.")
lines.append("")
lines.append("## 2. Included audit branches")
lines.append("")
lines.append("### Day21A")
lines.append("")
lines.append("- Protocol family: AC-off after Vmax")
lines.append("- Framework: A/B/D AC-off segmentation")
lines.append("- Group: 0.3C DC vs 0.3C + 0.7C")
lines.append("- κ = 2.33")
lines.append("- Frequencies: 0.1τ, 1τ, 10τ")
lines.append("")
lines.append("### Day22A")
lines.append("")
lines.append("- Protocol family: AC-off after Vmax")
lines.append("- Framework: A/B/D AC-off segmentation")
lines.append("- Group: 0.3C DC vs 0.3C + 0.4C")
lines.append("- κ = 1.33")
lines.append("- Frequencies: 0.1τ, 1τ, 10τ")
lines.append("")
lines.append("### Day23A")
lines.append("")
lines.append("- Protocol family: continued AC after Vmax")
lines.append("- Framework: G0/G1/G2 generalized boundary ordering")
lines.append("- Group: 0.4C DC vs 0.4C + 0.1C / 0.4C + 0.3C")
lines.append("- κ = 0.25 and 0.75")
lines.append("- Frequency: fixed 1τ")
lines.append("- 0.4C + 0.2C excluded due to missing pre-Vmax CC raw segment")
lines.append("")
lines.append("## 3. Main MJ1 experimental synthesis table")
lines.append("")
lines.append("```text")
lines.append(summary_table)
lines.append("```")
lines.append("")
lines.append("## 4. Day-level summaries")
lines.append("")
lines.append("### Day21A — high-amplitude AC-off protocol family")
lines.append("")
lines.append("```text")
lines.append(day21_table)
lines.append("```")
lines.append("")
lines.append("Day21A shows that high-amplitude AC-off protocols are DCAC-first by Q. For 1τ and 10τ, both Q80_common and Q90_common lie in the boundary/control-state split region. The full-protocol raw gains are therefore best interpreted as boundary/control-state mediated rather than confirmed Segment-A non-geometric acceleration.")
lines.append("")
lines.append("### Day22A — lower-amplitude AC-off protocol family")
lines.append("")
lines.append("```text")
lines.append(day22_table)
lines.append("```")
lines.append("")
lines.append("Day22A shows that lowering AC amplitude weakens the boundary pathway. Q80_common returns to the shared prescribed-current region, while Q90_common remains in the boundary/control-state split region. The effect is reduced but not strictly absent.")
lines.append("")
lines.append("### Day23A — sub-DC-amplitude continued-AC protocol-mode contrast")
lines.append("")
lines.append("```text")
lines.append(day23_table)
lines.append("```")
lines.append("")
lines.append("Day23A is not the same protocol family as Day21A/Day22A. AC continues after Vmax, so the original A/B/D AC-off segmentation is disabled. Day23A uses G0/G1/G2 generalized boundary ordering.")
lines.append("")
lines.append("For κ = 0.25, Q80/Q90 common raw gains are within self-consistency resolution, giving an unresolved state-gain result.")
lines.append("")
lines.append("For κ = 0.75, DCAC is boundary-leading and has positive Q80_common raw gain, but Q90_common becomes negative. This establishes that boundary-leading does not imply full-protocol gain preservation.")
lines.append("")
lines.append("## 5. PyBaMM context rows")
lines.append("")
lines.append("The Day21A closure file included PyBaMM Day20B context rows. These are preserved separately and excluded from the main MJ1 experimental synthesis table.")
lines.append("")
lines.append("```text")
lines.append(pybamm_table)
lines.append("```")
lines.append("")
lines.append("## 6. Cross-audit interpretation")
lines.append("")
lines.append("The combined Day21A–Day23A result supports the following structure:")
lines.append("")
lines.append("1. Raw Δt(Q) gains are real first-passage differences, but they are not mechanism-pure.")
lines.append("2. In the AC-off protocol family, increasing AC amplitude moves Q80/Q90 common anchors into the boundary/control-state split region.")
lines.append("3. Lowering AC amplitude weakens this boundary pathway and can return Q80_common to the shared prescribed-current region.")
lines.append("4. In the continued-AC protocol-mode family, boundary-leading can occur even when AC_C < DC_C.")
lines.append("5. Boundary-leading alone is insufficient: Day23A κ = 0.75 shows positive G0/Q80 gain but negative Q90/G2 gain.")
lines.append("6. No closed audit confirms clean non-geometric Segment-A or G0 electrochemical acceleration.")
lines.append("")
lines.append("## 7. Allowed claims")
lines.append("")
lines.append("Allowed:")
lines.append("")
lines.append("1. Day21A and Day22A support a boundary/control-state mediated interpretation within the AC-off protocol family.")
lines.append("2. Day22A shows that lower AC amplitude weakens the boundary pathway relative to Day21A.")
lines.append("3. Day23A shows that continued-AC protocols require a different G0/G1/G2 framework.")
lines.append("4. Day23A shows that DCAC-first boundary ordering can occur even under sub-DC AC amplitude.")
lines.append("5. Day23A also shows that boundary-leading does not guarantee full-protocol gain preservation.")
lines.append("6. Across all three audits, raw Δt(Q) must be interpreted by protocol region and protocol mode.")
lines.append("")
lines.append("## 8. Prohibited claims")
lines.append("")
lines.append("Do not claim:")
lines.append("")
lines.append("1. Segment-A or G0 non-geometric electrochemical acceleration has been confirmed.")
lines.append("2. Day23A is directly comparable to Day21A/Day22A as the same AC-off protocol family.")
lines.append("3. Boundary-leading is equivalent to full-protocol acceleration.")
lines.append("4. Q90/G2 raw gains in Day23A are late-CV preservation in the Day21A/Day22A sense.")
lines.append("5. Temperature effects can be quantified in Day23A; temperature summaries are missing.")
lines.append("")
lines.append("## 9. Key output files")
lines.append("")
lines.append(f"- Main MJ1 synthesis table: `{OUT_CROSS_SYNTHESIS_TABLE}`")
lines.append(f"- Excluded PyBaMM context table: `{OUT_CROSS_PYBAMM_CONTEXT}`")
lines.append(f"- Source inventory: `{OUT_CROSS_SOURCE_INVENTORY}`")
lines.append("")
lines.append("## 10. Closure")
lines.append("")
lines.append("This notebook closes the Day21A–Day23A cross-audit synthesis layer.")
lines.append("")
lines.append("Next recommended step:")
lines.append("")
lines.append("Use the synthesis table as the source for cross-audit visualization and documentation.")

synthesis_md = "\n".join(lines)

OUT_CROSS_SYNTHESIS_MD.write_text(synthesis_md, encoding="utf-8")

print(f"[OK] Wrote cross-audit synthesis note: {OUT_CROSS_SYNTHESIS_MD}")
print("[OK] Cell 3 cross-audit synthesis note completed.")
print("[OK] No closed audit verdict was modified.")

[OK] Wrote cross-audit synthesis note: /Users/louislu/pybamm-dcac-superimposed/data/day21A_day22A_day23A_cross_audit_synthesis_note.md
[OK] Cell 3 cross-audit synthesis note completed.
[OK] No closed audit verdict was modified.


In [5]:
# Notebook 28 Cell 4 — compact display / plotting table
#
# Purpose:
# - Generate a compact plotting/documentation table from the MJ1 synthesis table
# - Keep only high-value comparison columns
# - Add short labels for regions, protocol family, and figure grouping
#
# Explicitly NOT done here:
# - No recomputation
# - No new thresholds
# - No verdict modification

OUT_CROSS_COMPACT_TABLE = DATA_DIR / "day21A_day22A_day23A_cross_audit_compact_table.csv"

if not OUT_CROSS_SYNTHESIS_TABLE.exists():
    raise FileNotFoundError(f"Missing main synthesis table: {OUT_CROSS_SYNTHESIS_TABLE}")

df_cross_main = pd.read_csv(OUT_CROSS_SYNTHESIS_TABLE)


# =============================================================================
# 4.1 Label helpers
# =============================================================================

def short_region_label(region):
    s = str(region)

    if s.startswith("A_"):
        return "A"
    if s.startswith("B_"):
        return "B"
    if s.startswith("D_"):
        return "D"
    if s.startswith("G0_"):
        return "G0"
    if s.startswith("G1_"):
        return "G1"
    if s.startswith("G2_"):
        return "G2"
    if "outside" in s:
        return "outside"
    if s.strip() == "" or s.lower() == "nan":
        return "NA"

    return s


def short_family_label(row):
    day = row["audit_day"]
    family = row["protocol_family"]

    if family == "AC_off_after_Vmax":
        if day == "Day21A":
            return "AC-off, high amplitude"
        if day == "Day22A":
            return "AC-off, lower amplitude"
        return "AC-off"

    if family == "continued_AC_after_Vmax":
        return "continued AC after Vmax"

    return str(family)


def protocol_display_label(pair):
    s = str(pair)
    s = s.replace("0.3C DC vs ", "")
    s = s.replace("0.4C DC vs ", "")
    return s


def short_verdict_label(row):
    verdict = str(row["mechanism_verdict"])
    interp = str(row["synthesis_interpretation"])

    if "boundary_control_state_dominant" in interp:
        return "boundary/control-state dominant"

    if "weakens_boundary_pathway" in interp:
        return "weakened boundary pathway"

    if "gain_unresolved" in interp:
        return "unresolved near resolution"

    if "not_gain_preserving" in interp:
        return "boundary-leading, not preserved"

    if "ambiguous" in verdict:
        return "ambiguous"

    return verdict


def signed_gain_label(x):
    try:
        val = float(x)
    except Exception:
        return "NA"

    if val > 0:
        return "positive"
    if val < 0:
        return "negative"
    return "zero"


def figure_group_label(row):
    day = row["audit_day"]

    if day == "Day21A":
        return "A_high_amp_ACoff"
    if day == "Day22A":
        return "B_lower_amp_ACoff"
    if day == "Day23A":
        return "C_subDC_continuedAC"

    return "unknown"


# =============================================================================
# 4.2 Build compact table
# =============================================================================

compact_rows = []

for _, row in df_cross_main.iterrows():
    compact_rows.append({
        "audit_day": row["audit_day"],
        "figure_group": figure_group_label(row),
        "protocol_display": protocol_display_label(row["protocol_pair"]),
        "protocol_pair": row["protocol_pair"],

        "protocol_family_short": short_family_label(row),
        "framework": row["framework"],

        "DC_C": row["DC_C"],
        "AC_C": row["AC_C"],
        "kappa": row["kappa"],
        "m_tau": row["m_tau"],

        "AC_off_after_Vmax_status": row["AC_off_after_Vmax_status"],
        "boundary_ordering_by_Q": row["boundary_ordering_by_Q"],
        "Q_Vmax_shift_Ah_DC_minus_DCAC": row["Q_Vmax_shift_Ah_DC_minus_DCAC"],

        "Q80_region_short": short_region_label(row["Q80_common_region"]),
        "Q90_region_short": short_region_label(row["Q90_common_region"]),
        "Q80_common_region": row["Q80_common_region"],
        "Q90_common_region": row["Q90_common_region"],

        "dt_Q80_common_raw_s": row["dt_Q80_common_raw_s"],
        "dt_Q90_common_raw_s": row["dt_Q90_common_raw_s"],
        "Q80_raw_gain_sign": signed_gain_label(row["dt_Q80_common_raw_s"]),
        "Q90_raw_gain_sign": signed_gain_label(row["dt_Q90_common_raw_s"]),

        "residual_region_name": row["residual_region_name"],
        "prescribed_residual_status": row["prescribed_residual_status"],
        "prescribed_residual_p95_abs_s": row["prescribed_residual_p95_abs_s"],
        "fitted_residual_p95_abs_s": row["fitted_residual_p95_abs_s"],

        "evidence_status": row["evidence_status"],
        "mechanism_verdict": row["mechanism_verdict"],
        "verdict_short": short_verdict_label(row),
        "synthesis_interpretation": row["synthesis_interpretation"],

        "same_protocol_family_as_Day21A_Day22A": row["same_protocol_family_as_Day21A_Day22A"],
        "caveat": row["caveat"],
    })

df_compact = pd.DataFrame(compact_rows)

# Fixed visual sort
day_order = {
    "Day21A": 1,
    "Day22A": 2,
    "Day23A": 3,
}

df_compact["_day_order"] = df_compact["audit_day"].map(day_order)
df_compact = df_compact.sort_values(
    by=["_day_order", "m_tau", "kappa"],
    ascending=[True, True, True],
).drop(columns=["_day_order"])

df_compact.to_csv(OUT_CROSS_COMPACT_TABLE, index=False)

print(f"[OK] Wrote compact cross-audit table: {OUT_CROSS_COMPACT_TABLE}")
print(f"[OK] compact rows = {len(df_compact)}")

display_cols = [
    "audit_day",
    "protocol_display",
    "protocol_family_short",
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "boundary_ordering_by_Q",
    "Q_Vmax_shift_Ah_DC_minus_DCAC",
    "Q80_region_short",
    "Q90_region_short",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "verdict_short",
]

print(df_compact[display_cols].to_string(index=False))


# =============================================================================
# 4.3 Hard guards
# =============================================================================

if len(df_compact) != 8:
    raise ValueError(f"Expected 8 compact MJ1 rows, got {len(df_compact)}")

if not set(df_compact["audit_day"].unique()) == {"Day21A", "Day22A", "Day23A"}:
    raise ValueError("Compact table does not contain exactly Day21A, Day22A, Day23A.")

if df_compact["protocol_pair"].duplicated().any():
    dupes = df_compact.loc[df_compact["protocol_pair"].duplicated(), "protocol_pair"].tolist()
    raise ValueError(f"Duplicate protocol_pair rows in compact table: {dupes}")

print("[OK] Cell 4 compact display / plotting table completed.")
print("[OK] This table is ready for figures and docs.")

[OK] Wrote compact cross-audit table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_day22A_day23A_cross_audit_compact_table.csv
[OK] compact rows = 8
audit_day protocol_display   protocol_family_short  DC_C  AC_C    kappa  m_tau boundary_ordering_by_Q  Q_Vmax_shift_Ah_DC_minus_DCAC Q80_region_short Q90_region_short  dt_Q80_common_raw_s  dt_Q90_common_raw_s                   verdict_short
   Day21A 0.3C+0.7C 0.1tau  AC-off, high amplitude   0.3   0.7 2.333333    0.1             DCAC_first                       0.288741                A                B             0.196020           351.715593                       ambiguous
   Day21A   0.3C+0.7C 1tau  AC-off, high amplitude   0.3   0.7 2.333333    1.0             DCAC_first                       0.412826                B                B           139.916741           476.912802 boundary/control-state dominant
   Day21A  0.3C+0.7C 10tau  AC-off, high amplitude   0.3   0.7 2.333333   10.0             DCAC_first                    

In [6]:
# Notebook 28 Cell 5 — inject key conclusion into synthesis note
#
# Purpose:
# - Add a compact key conclusion to the cross-audit synthesis note
# - Preserve existing synthesis note content
# - No recomputation
# - No verdict modification

if not OUT_CROSS_SYNTHESIS_MD.exists():
    raise FileNotFoundError(f"Missing synthesis note: {OUT_CROSS_SYNTHESIS_MD}")

note_text = OUT_CROSS_SYNTHESIS_MD.read_text(encoding="utf-8")

key_conclusion_block = """## Key conclusion

The Day21A–Day23A audits jointly show that earlier Vmax triggering under DC–AC excitation can be a real boundary event. However, boundary-leading does not automatically imply full-protocol state-equivalent gain preservation, and it does not constitute confirmed non-geometric Segment-A/G0 electrochemical acceleration.

"""

if "## Key conclusion" not in note_text:
    # Insert after the first metadata block, before Scope if possible.
    marker = "## 1. Scope"
    if marker in note_text:
        note_text = note_text.replace(marker, key_conclusion_block + marker)
    else:
        note_text = key_conclusion_block + note_text

    OUT_CROSS_SYNTHESIS_MD.write_text(note_text, encoding="utf-8")
    print(f"[OK] Injected key conclusion into: {OUT_CROSS_SYNTHESIS_MD}")
else:
    print("[OK] Key conclusion already exists. No change made.")

print("[OK] Cell 5 completed. No recomputation or verdict modification performed.")

[OK] Injected key conclusion into: /Users/louislu/pybamm-dcac-superimposed/data/day21A_day22A_day23A_cross_audit_synthesis_note.md
[OK] Cell 5 completed. No recomputation or verdict modification performed.
